# GlowLab · 04 — ONNX ga eksport

PyTorch modeli Electron ichida ishlamaydi → **ONNX** formatiga o'tkazamiz va ilovada `onnxruntime-node` bilan **oflayn** ishlatamiz.
Natija fayllar: `product_classifier.onnx`, `classes.json`, `preprocess.json` → `glowlab/resources/models/` ga ko'chiriladi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/glowlab_ml'
DATA, OUT = f'{ROOT}/data', f'{ROOT}/models'

!pip install -q onnx onnxruntime    # Eksportni tekshirish uchun

In [ ]:
import json, os
import numpy as np
import torch, torch.nn as nn
from torchvision import models, transforms
from PIL import Image

CLASSES = json.load(open(f'{OUT}/classes.json'))
metrics = json.load(open(f'{OUT}/metrics.json'))
model = models.mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(model.last_channel, len(CLASSES))
model.load_state_dict(torch.load(f'{OUT}/product_classifier.pt', map_location='cpu'))
model.eval()                                   # Dropout/BatchNorm bashorat rejimida

In [ ]:
dummy = torch.randn(1, 3, 224, 224)            # Namuna kirish: 1 rasm, RGB, 224x224
torch.onnx.export(
    model, dummy, f'{OUT}/product_classifier.onnx',
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},   # Batch o'lchami o'zgaruvchan
    opset_version=17,
    dynamo=False,     # Klassik eksporter: dynamic_axes bilan barqaror ishlaydi
)
print('onnx size MB:', round(os.path.getsize(f'{OUT}/product_classifier.onnx') / 1e6, 1))

In [ ]:
# Ilova shu fayldan preprocessing va threshold'ni o'qiydi — Python bilan AYNAN bir xil bo'lishi shart
preprocess = {
    'size': 224,
    'mean': [0.485, 0.456, 0.406],
    'std': [0.229, 0.224, 0.225],
    'layout': 'NCHW',                     # [batch, kanal, balandlik, kenglik]
    'threshold': metrics['threshold'],    # 03-notebook'da val'dan tanlangan
}
json.dump(preprocess, open(f'{OUT}/preprocess.json', 'w'), indent=2)
print(preprocess)

## Tekshiruv: ONNX va PyTorch bir xil javob beradimi?

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(f'{OUT}/product_classifier.onnx')
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize(preprocess['mean'], preprocess['std'])])

ok, total = 0, 0
for c in CLASSES:
    for name in sorted(os.listdir(f'{DATA}/test/{c}'))[:5]:          # Har sinfdan 5 ta
        x = tf(Image.open(f'{DATA}/test/{c}/{name}').convert('RGB')).unsqueeze(0)
        with torch.no_grad():
            torch_out = model(x).numpy()
        onnx_out = sess.run(['logits'], {'input': x.numpy()})[0]
        ok += int(torch_out.argmax() == onnx_out.argmax())
        total += 1
        assert np.allclose(torch_out, onnx_out, atol=1e-3), 'farq katta!'
print(f'top-1 mos keldi: {ok}/{total}')   # 30/30 bo'lishi kerak

In [ ]:
# Ilovaga kerakli fayllarni yuklab olish (Mac'ga tushadi → glowlab/resources/models/ ga ko'chiring)
from google.colab import files
for name in ['product_classifier.onnx', 'classes.json', 'preprocess.json', 'metrics.json', 'confusion_matrix.png']:
    files.download(f'{OUT}/{name}')